# One-click PlantVillage ResNet-50 experiment

This Google Colab notebook implements the experimental workflow described in **Automated Plant Disease Detection and Diagnosis Using Deep Learning Architectures**.

It automatically:

1. mounts Google Drive when available;
2. downloads the public PlantVillage dataset;
3. creates a reproducible stratified 70/15/15 split;
4. loads an existing ResNet-50 checkpoint if one is found;
5. otherwise trains for up to 25 epochs and saves the best checkpoint;
6. reloads the best model and generates the complete **Results and Analysis** outputs:
   - weighted accuracy, precision, recall, and F1-score;
   - class distribution chart;
   - normalized 38-class confusion matrix;
   - training and validation curves;
   - paper-specific confusion rates;
   - Grad-CAM prediction examples;
   - parameter count and measured inference latency;
   - CSV, JSON, Markdown, model checkpoint, and ZIP exports.

**Single-click use:** open this file in Colab, choose a GPU runtime, then select **Runtime > Run all**. Google Drive may ask for one authorization click. A complete first run trains the model; later runs automatically load the saved checkpoint.

> Important: The notebook measures results from the actual run. It does not force the paper's reported 98.42% accuracy or the stated confusion percentages.


In [ ]:
# Install only the small dependencies that may be absent from Colab.
import importlib.util
import subprocess
import sys

required = {
    "kagglehub": "kagglehub>=0.3.7",
    "seaborn": "seaborn>=0.13",
    "sklearn": "scikit-learn>=1.3",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies ready.")


In [ ]:
# Configuration and reproducibility
import json
import math
import os
import random
import shutil
import time
import warnings
from collections import Counter
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from torchvision.models import ResNet50_Weights

warnings.filterwarnings("ignore")

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
MAX_EPOCHS = 25
EARLY_STOPPING_PATIENCE = 7
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
USE_GOOGLE_DRIVE = True
AUTO_TRAIN_IF_MISSING = True
DATASET_HANDLE = "abdallahalidev/plantvillage-dataset"
CHECKPOINT_FILENAME = "best_resnet50_plantvillage.pth"

# Optional: provide a direct downloadable .pth/.pt URL. Leave blank to use Drive or train.
MODEL_URL = ""

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

drive_mounted = False
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        drive_mounted = True
    except Exception as exc:
        print("Google Drive not mounted; using local /content storage:", exc)

base_dir = Path("/content/drive/MyDrive/PlantDiseaseResNet50" if drive_mounted else "/content/PlantDiseaseResNet50")
output_dir = base_dir / "results"
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = base_dir / CHECKPOINT_FILENAME
print("Checkpoint:", checkpoint_path)
print("Outputs:", output_dir)


In [ ]:
# Download and locate the 38-class color PlantVillage dataset.
dataset_download = Path(kagglehub.dataset_download(DATASET_HANDLE))
print("Downloaded dataset root:", dataset_download)

image_suffixes = {".jpg", ".jpeg", ".png", ".bmp"}

def valid_imagefolder_root(path):
    if not path.is_dir():
        return False
    class_dirs = [child for child in path.iterdir() if child.is_dir()]
    if len(class_dirs) != 38:
        return False
    return all(any(f.suffix.lower() in image_suffixes for f in d.iterdir() if f.is_file()) for d in class_dirs)

preferred = [
    dataset_download / "plantvillage dataset" / "color",
    dataset_download / "PlantVillage-Dataset" / "raw" / "color",
    dataset_download / "color",
]
dataset_dir = next((p for p in preferred if valid_imagefolder_root(p)), None)
if dataset_dir is None:
    dataset_dir = next((p for p in dataset_download.rglob("*") if valid_imagefolder_root(p)), None)
if dataset_dir is None:
    raise FileNotFoundError("Could not locate a 38-class ImageFolder directory in the downloaded dataset.")

print("Using image directory:", dataset_dir)

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

index_dataset = datasets.ImageFolder(dataset_dir)
targets = np.asarray(index_dataset.targets)
all_indices = np.arange(len(index_dataset))

train_indices, temp_indices = train_test_split(
    all_indices,
    test_size=0.30,
    random_state=SEED,
    stratify=targets,
)
val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=SEED,
    stratify=targets[temp_indices],
)

train_dataset = Subset(datasets.ImageFolder(dataset_dir, transform=train_transform), train_indices)
val_dataset = Subset(datasets.ImageFolder(dataset_dir, transform=eval_transform), val_indices)
test_dataset = Subset(datasets.ImageFolder(dataset_dir, transform=eval_transform), test_indices)

class_names = index_dataset.classes
class_to_idx = index_dataset.class_to_idx
num_classes = len(class_names)
assert num_classes == 38, f"Expected 38 classes, found {num_classes}"

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda")
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print(f"Total images: {len(index_dataset):,}")
print(f"Train/validation/test: {len(train_dataset):,}/{len(val_dataset):,}/{len(test_dataset):,}")
print(f"Classes: {num_classes}")

# Figure 3 in the paper: class distribution.
def class_counts(indices):
    counts = Counter(targets[indices].tolist())
    return [counts.get(i, 0) for i in range(num_classes)]

distribution_df = pd.DataFrame({
    "Class": class_names,
    "Train": class_counts(train_indices),
    "Validation": class_counts(val_indices),
    "Test": class_counts(test_indices),
})
distribution_df.to_csv(output_dir / "class_distribution.csv", index=False)

ax = distribution_df.set_index("Class")[["Train", "Validation", "Test"]].plot(
    kind="bar", stacked=True, figsize=(18, 8), width=0.9
)
ax.set_title("PlantVillage stratified 70/15/15 class distribution")
ax.set_ylabel("Number of images")
ax.set_xlabel("Class")
ax.tick_params(axis="x", labelsize=7)
plt.tight_layout()
plt.savefig(output_dir / "class_distribution.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Model, checkpoint, and automatic load/train functions.
def build_model(num_classes=38, pretrained=True):
    weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    model = models.resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def clean_state_dict(state_dict):
    return {key.removeprefix("module."): value for key, value in state_dict.items()}

def load_checkpoint(model, path):
    payload = torch.load(path, map_location=device)
    if isinstance(payload, dict) and "model_state_dict" in payload:
        state_dict = payload["model_state_dict"]
    elif isinstance(payload, dict) and "state_dict" in payload:
        state_dict = payload["state_dict"]
    else:
        state_dict = payload
    model.load_state_dict(clean_state_dict(state_dict), strict=True)
    history = payload.get("history", {}) if isinstance(payload, dict) else {}
    saved_classes = payload.get("class_names") if isinstance(payload, dict) else None
    if saved_classes is not None and list(saved_classes) != list(class_names):
        raise ValueError("Checkpoint class order does not match the downloaded dataset.")
    return history, payload

def download_model_if_configured(destination):
    if not MODEL_URL or destination.exists():
        return
    import urllib.request
    print("Downloading configured model checkpoint...")
    urllib.request.urlretrieve(MODEL_URL, destination)

def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    running_loss = 0.0
    correct = 0
    total = 0
    amp_enabled = device.type == "cuda"

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            logits = model(images)
            loss = criterion(logits, labels)

        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

def train_model(model):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.2, patience=2
    )
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, scaler)
        with torch.no_grad():
            val_loss, val_acc = run_epoch(model, val_loader, criterion)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        print(
            f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
            f"train loss {train_loss:.4f}, acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f}, acc {val_acc:.4f}"
        )

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "class_names": class_names,
                "class_to_idx": class_to_idx,
                "history": history,
                "best_val_loss": best_val_loss,
                "epoch": epoch,
                "config": {
                    "seed": SEED,
                    "image_size": IMAGE_SIZE,
                    "batch_size": BATCH_SIZE,
                    "max_epochs": MAX_EPOCHS,
                    "learning_rate": LEARNING_RATE,
                    "weight_decay": WEIGHT_DECAY,
                },
            }, checkpoint_path)
            print("  Saved best checkpoint:", checkpoint_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print("Early stopping triggered.")
                break
    return history

download_model_if_configured(checkpoint_path)
model = build_model(num_classes=num_classes, pretrained=not checkpoint_path.exists()).to(device)

if checkpoint_path.exists():
    print("Loading existing checkpoint:", checkpoint_path)
    history, checkpoint = load_checkpoint(model, checkpoint_path)
    model_source = "loaded"
elif AUTO_TRAIN_IF_MISSING:
    print("No checkpoint found. Training automatically...")
    history = train_model(model)
    model = build_model(num_classes=num_classes, pretrained=False).to(device)
    history, checkpoint = load_checkpoint(model, checkpoint_path)
    model_source = "trained_then_loaded"
else:
    raise FileNotFoundError(
        f"No checkpoint found at {checkpoint_path}. Set AUTO_TRAIN_IF_MISSING=True, upload a checkpoint, or set MODEL_URL."
    )

model.eval()
print("Model ready:", model_source)


In [ ]:
# Complete test-set evaluation for Results and Analysis.
@torch.inference_mode()
def predict_loader(model, loader):
    y_true, y_pred, y_prob, paths = [], [], [], []
    base_samples = datasets.ImageFolder(dataset_dir).samples
    test_paths = [base_samples[i][0] for i in test_indices]
    offset = 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        logits = model(images)
        probabilities = logits.softmax(dim=1)
        predictions = probabilities.argmax(dim=1)
        batch_n = labels.size(0)
        y_true.extend(labels.numpy().tolist())
        y_pred.extend(predictions.cpu().numpy().tolist())
        y_prob.extend(probabilities.cpu().numpy().tolist())
        paths.extend(test_paths[offset:offset + batch_n])
        offset += batch_n
    return np.asarray(y_true), np.asarray(y_pred), np.asarray(y_prob), paths

y_true, y_pred, y_prob, test_paths = predict_loader(model, test_loader)
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

metrics = {
    "accuracy": float(accuracy),
    "precision_weighted": float(precision),
    "recall_weighted": float(recall),
    "f1_weighted": float(f1),
    "test_images": int(len(y_true)),
    "model_source": model_source,
    "checkpoint": str(checkpoint_path),
}
print(json.dumps(metrics, indent=2))
(output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))

report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T
report_df.to_csv(output_dir / "classification_report.csv")
display(report_df.round(4))


In [ ]:
# Figure 4 in the paper: normalized confusion matrix and named confusion pairs.
cm_counts = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))
row_totals = cm_counts.sum(axis=1, keepdims=True)
cm_normalized = np.divide(cm_counts, row_totals, where=row_totals != 0) * 100.0

pd.DataFrame(cm_counts, index=class_names, columns=class_names).to_csv(
    output_dir / "confusion_matrix_counts.csv"
)
pd.DataFrame(cm_normalized, index=class_names, columns=class_names).to_csv(
    output_dir / "confusion_matrix_percent.csv"
)

plt.figure(figsize=(24, 20))
sns.heatmap(
    cm_normalized,
    cmap="Blues",
    vmin=0,
    vmax=100,
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={"label": "Percentage of true class"},
)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("Normalized confusion matrix - ResNet-50 on PlantVillage test set")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(output_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

def find_class_index(*fragments):
    for i, name in enumerate(class_names):
        normalized = name.lower().replace("_", " ")
        if all(fragment.lower() in normalized for fragment in fragments):
            return i
    return None

def directed_confusion(true_index, predicted_index):
    if true_index is None or predicted_index is None or cm_counts[true_index].sum() == 0:
        return float("nan")
    return 100.0 * cm_counts[true_index, predicted_index] / cm_counts[true_index].sum()

early = find_class_index("tomato", "early", "blight")
late = find_class_index("tomato", "late", "blight")
gray = find_class_index("corn", "gray", "leaf", "spot")
northern = find_class_index("corn", "northern", "leaf", "blight")

pair_results = {
    "Tomato Early Blight -> Late Blight (%)": directed_confusion(early, late),
    "Tomato Late Blight -> Early Blight (%)": directed_confusion(late, early),
    "Corn Gray Leaf Spot -> Northern Leaf Blight (%)": directed_confusion(gray, northern),
    "Corn Northern Leaf Blight -> Gray Leaf Spot (%)": directed_confusion(northern, gray),
}
pair_df = pd.DataFrame([pair_results]).T.rename(columns={0: "Measured value"})
pair_df.to_csv(output_dir / "paper_specific_confusions.csv")
display(pair_df.round(3))


In [ ]:
# Figure 6 in the paper: training and validation curves.
if history and history.get("train_loss"):
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].plot(epochs, history["train_loss"], marker="o", label="Training loss")
    axes[0].plot(epochs, history["val_loss"], marker="o", label="Validation loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Cross-entropy loss")
    axes[0].set_title("Loss curves")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(epochs, np.asarray(history["train_acc"]) * 100, marker="o", label="Training accuracy")
    axes[1].plot(epochs, np.asarray(history["val_acc"]) * 100, marker="o", label="Validation accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].set_title("Accuracy curves")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(output_dir / "training_curves.png", dpi=300, bbox_inches="tight")
    plt.show()

    pd.DataFrame({"epoch": epochs, **history}).to_csv(output_dir / "training_history.csv", index=False)
else:
    print("This checkpoint does not contain training history; training_curves.png cannot be recreated.")


In [ ]:
# Figure 5 in the paper: Grad-CAM examples, prioritizing misclassified samples.
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.forward_handle = target_layer.register_forward_hook(self._save_activation)

    def _save_activation(self, module, inputs, output):
        self.activations = output.detach()
        output.register_hook(self._save_tensor_gradient)

    def _save_tensor_gradient(self, gradient):
        self.gradients = gradient.detach()

    def __call__(self, image, target_class=None):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(image)
        if target_class is None:
            target_class = int(logits.argmax(1).item())
        logits[:, target_class].sum().backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1).relu()
        cam = torch.nn.functional.interpolate(
            cam.unsqueeze(1), size=image.shape[-2:], mode="bilinear", align_corners=False
        ).squeeze()
        cam -= cam.min()
        cam /= cam.max().clamp_min(1e-8)
        return cam.cpu().numpy(), logits.softmax(1).detach().cpu().numpy()[0]

    def close(self):
        self.forward_handle.remove()

def denormalize(image_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    return (image_tensor.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

wrong = np.flatnonzero(y_true != y_pred).tolist()
correct = np.flatnonzero(y_true == y_pred).tolist()
rng = np.random.default_rng(SEED)
rng.shuffle(wrong)
rng.shuffle(correct)
selected = (wrong[:4] + correct[:2])[:6]

gradcam = GradCAM(model, model.layer4[-1])
fig, axes = plt.subplots(len(selected), 3, figsize=(12, 4 * len(selected)))
if len(selected) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, test_position in enumerate(selected):
    image, true_label = test_dataset[test_position]
    image_batch = image.unsqueeze(0).to(device)
    cam, probabilities = gradcam(image_batch)
    predicted_label = int(probabilities.argmax())
    rgb = denormalize(image)

    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title("Input image")
    axes[row, 1].imshow(cam, cmap="jet", vmin=0, vmax=1)
    axes[row, 1].set_title("Grad-CAM")
    axes[row, 2].imshow(rgb)
    axes[row, 2].imshow(cam, cmap="jet", alpha=0.45, vmin=0, vmax=1)
    status = "Correct" if predicted_label == true_label else "Incorrect"
    axes[row, 2].set_title(
        f"{status}\nTrue: {class_names[true_label]}\nPred: {class_names[predicted_label]} ({probabilities[predicted_label]:.2%})",
        fontsize=9,
    )
    for axis in axes[row]:
        axis.axis("off")

plt.tight_layout()
plt.savefig(output_dir / "gradcam_examples.png", dpi=300, bbox_inches="tight")
plt.show()
gradcam.close()


In [ ]:
# Parameter count, reproducible inference benchmark, and paper comparison table.
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

@torch.inference_mode()
def benchmark_latency(model, iterations=100, warmup=20):
    sample = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    for _ in range(warmup):
        model(sample)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iterations):
        model(sample)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return 1000.0 * (time.perf_counter() - start) / iterations

measured_latency_ms = benchmark_latency(model)
efficiency = {
    "total_parameters": int(total_parameters),
    "trainable_parameters": int(trainable_parameters),
    "parameters_millions": total_parameters / 1e6,
    "batch_1_latency_ms": measured_latency_ms,
    "device": str(device),
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
}
(output_dir / "efficiency.json").write_text(json.dumps(efficiency, indent=2))
print(json.dumps(efficiency, indent=2))

comparison_df = pd.DataFrame([
    {"Model": "SVM + HOG", "Accuracy (%)": 89.4, "Parameters (M)": np.nan, "Inference (ms)": 15, "Source": "paper table"},
    {"Model": "AlexNet", "Accuracy (%)": 97.1, "Parameters (M)": 60, "Inference (ms)": 25, "Source": "paper table"},
    {"Model": "VGG-16", "Accuracy (%)": 99.2, "Parameters (M)": 138, "Inference (ms)": 140, "Source": "paper table"},
    {"Model": "ResNet-50 (paper)", "Accuracy (%)": 98.4, "Parameters (M)": 25, "Inference (ms)": 45, "Source": "paper table"},
    {
        "Model": "ResNet-50 (this run)",
        "Accuracy (%)": 100 * accuracy,
        "Parameters (M)": total_parameters / 1e6,
        "Inference (ms)": measured_latency_ms,
        "Source": "measured by this notebook",
    },
])
comparison_df.to_csv(output_dir / "model_comparison.csv", index=False)
display(comparison_df.round(3))


In [ ]:
# Export a paper-ready summary and a ZIP containing all generated artifacts.
summary = f'''# Results and Analysis generated by the Colab notebook

## Performance metrics
- Overall accuracy: {accuracy * 100:.2f}%
- Weighted precision: {precision * 100:.2f}%
- Weighted recall: {recall * 100:.2f}%
- Weighted F1-score: {f1 * 100:.2f}%
- Test images: {len(y_true):,}

## Efficiency
- Parameters: {total_parameters / 1e6:.2f} million
- Batch-1 inference latency: {measured_latency_ms:.2f} ms
- Device: {device}
- GPU: {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'N/A'}

## Validation note
These values were measured by this notebook from the held-out stratified test split. They should replace, not be adjusted to match, any unverified placeholder values in the paper.
'''
(output_dir / "paper_results_summary.md").write_text(summary)
print(summary)

archive_base = str(base_dir / "plant_disease_results")
archive_path = shutil.make_archive(archive_base, "zip", output_dir)
print("Saved checkpoint:", checkpoint_path)
print("Saved results folder:", output_dir)
print("Saved ZIP archive:", archive_path)

try:
    from google.colab import files
    print("To download everything, run: files.download(archive_path)")
except Exception:
    pass


## Files produced for the paper

The notebook writes these artifacts to the `results` folder:

- `class_distribution.png` - replaces the Figure 3 placeholder;
- `confusion_matrix.png` - replaces the Figure 4 placeholder;
- `gradcam_examples.png` - replaces the optional Figure 5 placeholder;
- `training_curves.png` - replaces the Figure 6 placeholder;
- `metrics.json` and `classification_report.csv` - Results and Analysis metrics;
- `paper_specific_confusions.csv` - measured disease-pair confusion rates;
- `model_comparison.csv` and `efficiency.json` - comparison-table support;
- `paper_results_summary.md` - values ready to transfer into the manuscript;
- `best_resnet50_plantvillage.pth` - automatically loaded on later runs.

Use the generated values instead of manually forcing the numerical claims already printed in the paper.
